AttentionLab Data Cleaning and Feature Engineering

In [36]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/wilson/AttentionLab/data/raw_attentionlab_data.csv")
df.head()
df.shape

(30, 19)

In [37]:
# Descriptive statistics and information about the DataFrame
df.info()
df.columns
df.describe()

# Check for missing values
df.isnull().sum()

# Convert Data Column to datetime
df["post_date"] = pd.to_datetime(df["post_date"])
df.dtypes

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   video_id              30 non-null     int64
 1   platform              30 non-null     str  
 2   post_date             30 non-null     str  
 3   post_time             30 non-null     str  
 4   caption               30 non-null     str  
 5   hook_text             30 non-null     str  
 6   hook_type             30 non-null     str  
 7   topic                 30 non-null     str  
 8   format                30 non-null     str  
 9   video_length_seconds  30 non-null     int64
 10  editing_time_minutes  30 non-null     int64
 11  views                 30 non-null     int64
 12  likes                 30 non-null     int64
 13  comments              30 non-null     int64
 14  shares                30 non-null     int64
 15  saves                 30 non-null     int64
 16  followers_gained     

video_id                         int64
platform                           str
post_date               datetime64[us]
post_time                          str
caption                            str
hook_text                          str
hook_type                          str
topic                              str
format                             str
video_length_seconds             int64
editing_time_minutes             int64
views                            int64
likes                            int64
comments                         int64
shares                           int64
saves                            int64
followers_gained                 int64
revenue                          int64
notes                              str
dtype: object

In [ ]:
# Creating Business Metrics

starting_followers = 0 #started with nothing
df["total_followers"] = df["followers_gained"].cumsum() + starting_followers

# Calculate total engagement rate
df["total_engagement_rate"] = (df["likes"] + df["comments"] + df["shares"]) + df["saves"]

# Calculate engagement rate
df["engagement_rate"] = df["total_engagement_rate"] / df["total_followers"]

# Calculate save rate
df["save_rate"] = df["saves"] / df["total_followers"]

# Calculate share rate
df["share_rate"] = df["shares"] / df["total_followers"]

# Follower conversion rate
df["follower_conversion_rate"] = (df["followers_gained"] / df["views"])

# Views per minute worked
df["views_per_minute"] = df["views"] / df["editing_time_minutes"]

# Engagement per minute worked
df["engagement_per_minute"] = df["total_engagement_rate"] / df["editing_time_minutes"]

In [39]:
# Creating Time Features
df["day_of_week"] = df["post_date"].dt.day_name()
df["month"] = df["post_date"].dt.month_name()
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"])

# Length Bucket
df["length_bucket"] = pd.cut(
    df["video_length_seconds"],
    bins=[0,15,30,60,100],
    labels=[
        "Under 15 sec",
        "15-30 sec",
        "31-60 sec",
        "Over 60 sec"
    ]
)

In [ ]:
# Answering Business Questions

# What platform has the highest average number of views?
df.groupby("platform")["views"].mean()


platform
Instagram Reels    17677.777778
TikTok             34866.666667
YouTube Shorts     27277.777778
Name: views, dtype: float64

In [42]:
# What topic has the highest average views?
df.groupby("topic")["views"].mean().sort_values(ascending=False)


topic
Finance         42400.000000
AI              40660.000000
Productivity    29266.666667
Coding          28975.000000
Career          23800.000000
Marketing       22566.666667
Study Tips      22466.666667
Tech News       21366.666667
Business        16800.000000
Lifestyle       10200.000000
Name: views, dtype: float64

In [ ]:
# What format has the highest average engagement rate?
df.groupby("format")["engagement_rate"].mean().sort_values(ascending=False)

format
Review          6.964069
Tutorial        3.717982
Listicle        1.671684
Challenge       1.456601
Case Study      1.309597
Reaction        1.230635
Demo            1.153294
Storytelling    0.969807
Name: engagement_rate, dtype: float64

In [ ]:
# What hook type has the highest average engagement rate?
df.groupby("hook_type")["engagement_rate"].mean().sort_values(ascending=False)

hook_type
Shocking Fact       8.279775
Problem Solution    2.886059
Tutorial Hook       2.691523
Mistake             2.392592
Story Hook          1.554031
Hot Take            1.498795
Question            1.064816
Case Study          0.716561
Name: engagement_rate, dtype: float64

Initial Findings

1. It seems like Tiktok by far has the highest average views compared to the other platforms.

2. Review formats show strongest engagement rates compare the the rest of the formats.

3. The shocking facts outperformed all other hook type by a mile.

4. The best topic to do Finance and a close second would be Ai, then everything else.

In [51]:
df.to_csv("/Users/wilson/AttentionLab/data/cleaned_attentionlab_data.csv", index=False)
